# PharmaShield GCC — Layer 4: Ranking & Dual-Source Order Split
Plan §4.2, §4.4, §5.2 steps 7–8. Run notebooks 01 and 02 first (they fill `data/processed/suppliers_matrix.csv`). Team decisions L1–L5 are in `docs/DECISIONS.md`.

GenAI disclosure (plan §6.1): code generated with Claude and reviewed by the team.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'layer4_decision').exists())
sys.path.insert(0, str(ROOT / 'layer4_decision'))
from decision import engine
from decision.weights import MODES
m, tiers, sku = engine.load_inputs()

## Step 7 — weights per mode and AHP consistency

In [2]:
res = {mode: engine.run(m, tiers, sku, mode, lam=1.0, entropy_share=0.0) for mode in MODES}
print('AHP consistency ratio:', res['Normal']['ahp_consistency']['CR'])
pd.concat({mode: r['weights']['Used for ranking'] for mode, r in res.items()}, axis=1)

Running HiGHS 1.16.0 (git hash: f3f113d): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
AHP consistency ratio: 0.0


,Normal,Crisis,Cost Focus
criterion,,,
C1_landed_cost,0.10,0.00,0.20
C2_delivery_risk,0.30,0.30,0.30
C3_price_integrity,0.05,0.05,0.05
C4_compliance_risk,0.30,0.30,0.30
C5_route_exposure,0.25,0.35,0.15


### Why the entropy share is 0 by default (decision L5): entropy weights are shown for information

In [3]:
res['Normal']['entropy_table']

,criterion,varies_across_makers,entropy,divergence
0,C1_landed_cost,True,0.9603,0.0397
1,C2_delivery_risk,True,0.4364,0.5636
2,C3_price_integrity,False,NaN,0.0000
3,C4_compliance_risk,True,0.9469,0.0531
4,C5_route_exposure,True,0.4324,0.5676


## TOPSIS ranking (S = closeness to the ideal supplier; higher is better)

In [4]:
pd.DataFrame({mode: r['ranking']['S'] for mode, r in res.items()}).sort_values('Normal', ascending=False).round(3)

,Normal,Crisis,Cost Focus
manufacturer_id,,,
BPI LABS,0.736,0.700,0.787
GLAND PHARMA,0.736,0.700,0.787
APOTEX,0.694,0.649,0.758
NATCO PHARMA USA,0.694,0.649,0.758
TEYRO LABS,0.694,0.649,0.758
AVYXA PHARMA,0.654,0.858,0.493
TEVA PARENTERAL MEDICINES,0.632,0.591,0.708
HOSPIRA,0.586,0.518,0.681
EUGIA US,0.536,0.456,0.643


## Step 8 — order split for 10,000 vials (PuLP)

In [5]:
for mode, r in res.items():
    a = r['allocation']
    print(mode, r['diagnostics']['solver_status'], '| spot buy', r['diagnostics']['spot_buy_vials'])
    display(a[a['selected']][['manufacturer_id', 'factory_group', 'S', 'vials', 'share_pct', 'role']])

Normal Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.736406,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.694264,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.632286,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.585968,1000.0,10.0,Additional source


Crisis Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.700163,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.649258,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.590596,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.518483,1000.0,10.0,Additional source


Cost Focus Optimal | spot buy 0.0


,manufacturer_id,factory_group,S,vials,share_pct,role
3,BPI LABS,GLAND_VISAKHAPATNAM,0.786642,3000.0,30.0,Primary
1,APOTEX,TEYRO_ORAGADAM,0.757728,3000.0,30.0,Backup
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.707875,3000.0,30.0,Additional source
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.681129,1000.0,10.0,Additional source


## Sensitivity of the split to the risk weight λ (the evidence for why the dashboards have no dial)

In [6]:
pd.DataFrame({lam: engine.run(m, tiers, sku, 'Normal', lam=lam)['allocation'].set_index('manufacturer_id')['vials']
              for lam in (0.0, 0.5, 1.0, 2.0)}).loc[lambda d: d.sum(axis=1) > 0]

,0.0,0.5,1.0,2.0
manufacturer_id,,,,
ACCORD HEALTHCARE,3000.0,0.0,0.0,0.0
APOTEX,3000.0,3000.0,3000.0,3000.0
BPI LABS,3000.0,3000.0,3000.0,3000.0
EUGIA US,1000.0,0.0,0.0,0.0
HOSPIRA,0.0,1000.0,1000.0,1000.0
TEVA PARENTERAL MEDICINES,0.0,3000.0,3000.0,3000.0


## Plan Q7 and §4.2 checks: warm-backup share and the Tier 1 two-supplier rule
Warm-backup share = the part of the order placed with factories other than the main one (supply already approved and shipping).

In [7]:
pd.DataFrame({mode: {k: r['diagnostics'][k] for k in ['factories_used', 'meets_two_supplier_minimum',
                                                   'largest_factory_share_pct', 'warm_backup_share_pct',
                                                   'spot_buy_vials', 'monthly_spend_usd']}
              for mode, r in res.items()})

,Normal,Crisis,Cost Focus
factories_used,4,4,4
meets_two_supplier_minimum,True,True,True
largest_factory_share_pct,30.0,30.0,30.0
warm_backup_share_pct,70.0,70.0,70.0
spot_buy_vials,0.0,0.0,0.0
monthly_spend_usd,279546.0,279546.0,279546.0


## Stress test (decision S1): a natural disaster takes factories offline
The ranking does not change; only the order split re-solves. Normal mode, λ = 1.

In [8]:
rows = []
for o in engine.stress_outages(m):
    d = engine.run(m, tiers, sku, 'Normal', 1.0, 0.0, set(o['groups']))['diagnostics']
    rows.append({'offline': o['key'], 'factories offline': len(o['groups']), 'emergency vials': d['spot_buy_vials'],
                 'monthly cost (USD)': d['monthly_spend_usd'], 'warm-backup share %': d['warm_backup_share_pct']})
pd.DataFrame(rows).set_index('offline')

,factories offline,emergency vials,monthly cost (USD),warm-backup share %
offline,,,,
AVYXA_SWITZERLAND,1,0.0,279546.0,70.0
EUGIA_UNIT1_KOLTHUR,1,0.0,279546.0,70.0
FRESENIUS_UNKNOWN,1,0.0,279546.0,70.0
GLAND_VISAKHAPATNAM,1,0.0,279546.0,70.0
HOSPIRA_AUSTRALIA,1,0.0,279546.0,70.0
INTAS_PHARMEZ_AHMEDABAD,1,0.0,279546.0,70.0
PHARMACHEMIE_HAARLEM,1,0.0,279546.0,70.0
TEYRO_ORAGADAM,1,0.0,279546.0,70.0
COUNTRY_INDIA,4,0.0,2607053.7,70.0


## Synthetic view (decision S4)
Built by `synthetic/build_synthetic.py` from the approved ranges in `synthetic/reference/synthetic_ranges.csv`. The real view above never uses these values.

In [9]:
ms, _, _ = engine.load_inputs(engine.MATRIX_SYNTHETIC)
res_syn = {mode: engine.run(ms, tiers, sku, mode, lam=1.0, entropy_share=0.0) for mode in MODES}
for mode, r in res_syn.items():
    a = r['allocation']; d = r['diagnostics']
    print(mode, '| established factories', d['established_factories'], '| warm-backup share', d['warm_backup_share_pct'], '%')
    display(a[a['selected']][['manufacturer_id', 'factory_group', 'S', 'vials', 'share_pct', 'role']])

Normal | established factories 5 | warm-backup share 63.0 %


,manufacturer_id,factory_group,S,vials,share_pct,role
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.612576,3700.0,37.0,Primary
5,FRESENIUS KABI USA,FRESENIUS_UNKNOWN,0.685229,3300.0,33.0,Backup
6,GLAND PHARMA,GLAND_VISAKHAPATNAM,0.888016,1500.0,15.0,Additional source
1,APOTEX,TEYRO_ORAGADAM,0.812529,1500.0,15.0,Additional source


Crisis | established factories 5 | warm-backup share 63.0 %


,manufacturer_id,factory_group,S,vials,share_pct,role
9,TEVA PARENTERAL MEDICINES,PHARMACHEMIE_HAARLEM,0.576557,3700.0,37.0,Primary
5,FRESENIUS KABI USA,FRESENIUS_UNKNOWN,0.645025,3300.0,33.0,Backup
6,GLAND PHARMA,GLAND_VISAKHAPATNAM,0.840501,1500.0,15.0,Additional source
1,APOTEX,TEYRO_ORAGADAM,0.768689,1500.0,15.0,Additional source


Cost Focus | established factories 5 | warm-backup share 63.0 %


,manufacturer_id,factory_group,S,vials,share_pct,role
7,HOSPIRA,HOSPIRA_AUSTRALIA,0.684103,3700.0,37.0,Primary
5,FRESENIUS KABI USA,FRESENIUS_UNKNOWN,0.745765,3300.0,33.0,Backup
6,GLAND PHARMA,GLAND_VISAKHAPATNAM,0.939218,1500.0,15.0,Additional source
1,APOTEX,TEYRO_ORAGADAM,0.852593,1500.0,15.0,Additional source


In [10]:
rows = []
for o in engine.stress_outages(ms):
    d = engine.run(ms, tiers, sku, 'Normal', 1.0, 0.0, set(o['groups']))['diagnostics']
    rows.append({'offline': o['key'], 'emergency vials': d['spot_buy_vials'], 'monthly cost (USD)': d['monthly_spend_usd']})
pd.DataFrame(rows).set_index('offline')

,emergency vials,monthly cost (USD)
offline,,
AVYXA_SWITZERLAND,0.0,283425.02
EUGIA_UNIT1_KOLTHUR,0.0,283425.02
FRESENIUS_UNKNOWN,0.0,283770.20
GLAND_VISAKHAPATNAM,0.0,283516.22
HOSPIRA_AUSTRALIA,0.0,283588.56
INTAS_PHARMEZ_AHMEDABAD,0.0,283425.02
PHARMACHEMIE_HAARLEM,0.0,283425.02
TEYRO_ORAGADAM,0.0,283653.62
COUNTRY_INDIA,0.0,283744.82


## Numbers for the slides (plan §5.2 step 10: every slide metric must match a notebook output)

In [11]:
import json
c2 = json.loads((ROOT / 'layer3_engines/outputs/c2_metrics.json').read_text())
c3 = json.loads((ROOT / 'layer3_engines/outputs/c3_metrics.json').read_text())
syn = json.loads((ROOT / 'synthetic/outputs/synthetic_summary.json').read_text())
e = m[m['eligible_for_ranking'].astype(bool)]
nd, cr, sd = res['Normal']['diagnostics'], res['Crisis']['diagnostics'], res_syn['Normal']['diagnostics']
india = engine.run(m, tiers, sku, 'Normal', 1.0, 0.0, set(next(o for o in engine.stress_outages(m) if o['kind'] == 'country')['groups']))['diagnostics']
facts = {
    'Labelers found (openFDA NDC)': len(m), 'Ranked (sell finished vials)': len(e),
    'Independent factories behind them': e['source_plant_group'].nunique(),
    'Delivery model: training / test shipments': f"{c2['train_rows']:,} / {c2['test_rows']:,}",
    'Delivery model ROC-AUC: XGBoost vs logistic regression': f"{c2['xgboost_test']['roc_auc']:.3f} vs {c2['logistic_regression_test']['roc_auc']:.3f}",
    'Late shipments caught (recall)': f"{c2['xgboost_test']['recall_late']:.0%}",
    'Billing model: lines checked / flagged': f"{c3['lines_scored']:,} / {c3['lines_flagged']:,}",
    'AHP consistency ratio': res['Normal']['ahp_consistency']['CR'],
    'Real view, Normal: factories / largest share / warm-backup share': f"{nd['factories_used']} / {nd['largest_factory_share_pct']:.0f}% / {nd['warm_backup_share_pct']:.0f}%",
    'Real view, monthly cost (official US prices)': f"${nd['monthly_spend_usd']:,.0f}",
    'Real view, every Indian factory offline: monthly cost': f"${india['monthly_spend_usd']:,.0f}",
    'Synthetic view, Normal: factories / largest share / warm-backup share': f"{sd['factories_used']} / {sd['largest_factory_share_pct']:.0f}% / {sd['warm_backup_share_pct']:.0f}%",
    'Synthetic view: planted invoice errors caught': f"{syn['c3_injected_errors_caught']} of {syn['c3_injected_errors']}",
}
pd.Series(facts, name='value').to_frame()

,value
Labelers found (openFDA NDC),18
Ranked (sell finished vials),11
Independent factories behind them,8
Delivery model: training / test shipments,"7,771 / 2,545"
Delivery model ROC-AUC: XGBoost vs logistic regression,0.797 vs 0.786
Late shipments caught (recall),92%
Billing model: lines checked / flagged,"8,436 / 422"
AHP consistency ratio,0.0
"Real view, Normal: factories / largest share / warm-backup share",4 / 30% / 70%
"Real view, monthly cost (official US prices)","$279,546"
